# Lab 07 — ML Foundations: Bias, Variance, and Cross-Validation

In this lab you will observe overfitting and underfitting directly from model outputs, use cross-validation to select a model without touching the test set, and interpret the bias-variance tradeoff on a real clinical dataset.

You do not need to build models from scratch — the fitting code is provided. Your job is to read the output, complete specific lines, and interpret what you see.

**Concepts covered:** bias-variance tradeoff, overfitting, underfitting, train vs. test error, cross-validation for model selection, loss functions.

**Reference notebooks:**
- `working-sessions/ml_concepts/04_bias_variance_tradeoff.ipynb`
- `working-sessions/ml_concepts/05_overfitting_and_underfitting.ipynb`
- `working-sessions/ml_concepts/06_cross_validation.ipynb`
- `working-sessions/ml_concepts/07_loss_functions.ipynb`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.datasets import make_classification

from tkh_utils import (
    PALETTE, FONT, base_layout,
    check_answer, make_answer_key, make_grading_summary,
    load_heart_disease,
)

_ak = make_answer_key({
    'q1': 'B',
    'q2': 'B',
    'q3': 'B',
    'q4': 'C',
})

---
## Section A — Multiple Choice

Fill in each answer variable with the letter of the best answer (A, B, C, or D).

In [ ]:
# Q1 — A model scores 97% on training data and 61% on test data. This is a sign of:
#
# A) High bias — the model is too simple to capture any patterns
# B) High variance — the model memorized the training set and fails to generalize
# C) A well-calibrated model — 97% training accuracy is exactly what we want
# D) A data leakage problem — the test set was seen during training

q1_answer = "___"  # Replace with A, B, C, or D

assert q1_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q1_answer, _ak['q1']), \
    "Not quite — revisit working-sessions/ml_concepts/05_overfitting_and_underfitting.ipynb"
print("✓ Question 1 correct!")

In [ ]:
# Q2 — What does 5-fold cross-validation do?
#
# A) Trains the model 5 times on random 5% samples of the data
# B) Splits the training data into 5 equal parts, uses each as a validation
#    set once, and averages the resulting scores across all 5 splits
# C) Picks the top 5 features and trains a separate model on each
# D) Trains one model and reports the score from the best of the 5 runs

q2_answer = "___"  # Replace with A, B, C, or D

assert q2_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q2_answer, _ak['q2']), \
    "Not quite — revisit working-sessions/ml_concepts/06_cross_validation.ipynb"
print("✓ Question 2 correct!")

In [ ]:
# Q3 — You compare two models:
#   Model A: train accuracy = 91%,  test accuracy = 88%
#   Model B: train accuracy = 99%,  test accuracy = 63%
#
# Which would you deploy?
#
# A) Model B — 99% training accuracy is the goal
# B) Model A — the small gap between train and test means it generalizes reliably
# C) Model B — higher training accuracy always leads to better real-world performance
# D) Neither — both models are failing because test accuracy is below 90%

q3_answer = "___"  # Replace with A, B, C, or D

assert q3_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q3_answer, _ak['q3']), \
    "Not quite — which model can you trust to perform well on data it has never seen?"
print("✓ Question 3 correct!")

In [ ]:
# Q4 — The No Free Lunch theorem tells us:
#
# A) Every model needs more data to perform well — there are no shortcuts
# B) Simpler models always outperform complex ones when data is limited
# C) No single algorithm is best for every problem — performance depends on
#    the structure of the data
# D) Cross-validation always identifies the best model regardless of the algorithm

q4_answer = "___"  # Replace with A, B, C, or D

assert q4_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q4_answer, _ak['q4']), \
    "Not quite — revisit working-sessions/ml_concepts/09_the_no_free_lunch_theorem.ipynb"
print("✓ Question 4 correct!")

In [ ]:
make_grading_summary([
    (q1_answer, _ak['q1'], "Q1: Diagnosing overfitting from accuracy numbers"),
    (q2_answer, _ak['q2'], "Q2: What cross-validation does"),
    (q3_answer, _ak['q3'], "Q3: Choosing a model to deploy"),
    (q4_answer, _ak['q4'], "Q4: No Free Lunch theorem"),
], total=4)

---
## Section B — Coding Exercises

The model-fitting code is provided — run each cell and observe the output. Your job is to complete specific lines and interpret what you see.

Run the setup cell first.

In [ ]:
# Shared setup — run this before the exercises
np.random.seed(42)
X_b, y_b = make_classification(n_samples=500, n_features=10, n_informative=5,
                                n_redundant=2, random_state=42)
X_b_train, X_b_test, y_b_train, y_b_test = train_test_split(
    X_b, y_b, test_size=0.25, random_state=42)

scaler_b = StandardScaler()
X_b_train_sc = scaler_b.fit_transform(X_b_train)
X_b_test_sc  = scaler_b.transform(X_b_test)

K_VALUES = [1, 3, 7, 15, 40]

# Fit one KNN model per k value — train and test accuracy are computed for you
train_accs, test_accs = [], []
for k in K_VALUES:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_b_train_sc, y_b_train)
    train_accs.append(accuracy_score(y_b_train, m.predict(X_b_train_sc)))
    test_accs.append(accuracy_score(y_b_test,  m.predict(X_b_test_sc)))

print(f"{'k':>5}  {'Train acc':>10}  {'Test acc':>10}  {'Gap (train-test)':>16}")
for k, tr, te in zip(K_VALUES, train_accs, test_accs):
    print(f"{k:>5}  {tr:>10.3f}  {te:>10.3f}  {tr - te:>16.3f}")

### B1 — Identify overfitting from accuracy numbers

Look at the table printed above. One value of k produces a training accuracy that is much higher than its test accuracy — a clear sign the model memorized the training set.

In [ ]:
# Which k shows the clearest sign of overfitting?
# (Highest training accuracy combined with the largest train-test gap)
overfitting_k = None  # YOUR CODE — choose a value from K_VALUES

# What is the training accuracy at that k? (look it up from the lists above)
overfit_train_acc = None  # YOUR CODE — one line

# What is the test accuracy at that k?
overfit_test_acc = None  # YOUR CODE — one line

# ── Assertions ───────────────────────────────────────────────────────────────
assert overfitting_k is not None, "Fill in overfitting_k"
assert overfitting_k == 1, \
    "Not quite — look for the k where training accuracy is near perfect but test accuracy drops"
assert overfit_train_acc is not None and overfit_train_acc > 0.98, \
    "At k=1, training accuracy should be nearly 1.0 — the model sees every training point as its own neighbor"
assert overfit_test_acc is not None and overfit_train_acc - overfit_test_acc > 0.10, \
    "The train-test gap at k=1 should be large — compute it from train_accs and test_accs"
print(f"✓ B1 complete!")
print(f"  k={overfitting_k}: train={overfit_train_acc:.3f}, test={overfit_test_acc:.3f}, "
      f"gap={overfit_train_acc - overfit_test_acc:.3f}")

### B2 — Use cross-validation to pick the best k

The code below runs 5-fold cross-validation for each k and prints the result. Cross-validation uses only the training set — the test set stays untouched until the very end.

Your job: identify the best k from the CV scores.

In [ ]:
# 5-fold CV accuracy for each k — this code is provided
cv_scores = []
for k in K_VALUES:
    scores = cross_val_score(KNeighborsClassifier(n_neighbors=k),
                             X_b_train_sc, y_b_train, cv=5, scoring="accuracy")
    cv_scores.append(scores.mean())

print("5-fold CV accuracy by k:")
for k, s in zip(K_VALUES, cv_scores):
    print(f"  k = {k:>2}:  {s:.3f}")

# ── Your turn: find the best k ───────────────────────────────────────────────
# Hint: np.argmax(cv_scores) gives the index of the highest value in the list
best_k = None  # YOUR CODE — one line

# ── Assertions ───────────────────────────────────────────────────────────────
assert best_k is not None, "Set best_k using np.argmax(cv_scores)"
assert best_k in K_VALUES, "best_k should be a value from K_VALUES, not an index"
assert best_k != 1, \
    "k=1 overfits the training data — cross-validation should penalize that"
print(f"\n✓ B2 complete! Best k by cross-validation: {best_k}")

### B3 — Complete the bias-variance visualization

The code below starts a plot with the training accuracy curve already drawn. Add the cross-validated accuracy line so we can compare the two.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(K_VALUES, train_accs, marker="o", linewidth=2, label="Train accuracy")

# YOUR CODE HERE — add the CV accuracy line
# ax.plot(K_VALUES, ???, marker="s", linewidth=2, linestyle="--", label="CV accuracy (5-fold)")


ax.axvline(best_k, color="crimson", linestyle=":", linewidth=1.5,
           label=f"Best k by CV = {best_k}")
ax.set_xlabel("Number of neighbors (k)")
ax.set_ylabel("Accuracy")
ax.set_title("KNN: Train vs. Cross-Validated Accuracy by k")
ax.legend()
plt.tight_layout()
plt.show()

# ── Assertions ───────────────────────────────────────────────────────────────
assert len(ax.lines) >= 3, \
    "Add a second line for CV accuracy — ax.plot(K_VALUES, cv_scores, ...)"
print("✓ B3 complete!")

---
## Section C — Applied Problem: Heart Disease Prediction

A hospital has data on 303 patients. The target is whether a patient has heart disease (1) or not (0), based on 13 clinical measurements.

The code below fits KNN models at several values of k, runs cross-validation, and records the results. Your job is to complete three targeted steps at the bottom.

In [ ]:
# Load and split the heart disease dataset
X_hd, y_hd = load_heart_disease()
X_hd_train, X_hd_test, y_hd_train, y_hd_test = train_test_split(
    X_hd, y_hd, test_size=0.25, random_state=42)

scaler_hd = StandardScaler()
X_hd_train_sc = scaler_hd.fit_transform(X_hd_train)
X_hd_test_sc  = scaler_hd.transform(X_hd_test)

K_HD = [1, 3, 5, 7, 11, 20, 35]

# Fit and record train accuracy, test accuracy, and CV accuracy for each k
hd_train_accs, hd_test_accs, hd_cv_accs = [], [], []
for k in K_HD:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_hd_train_sc, y_hd_train)
    hd_train_accs.append(accuracy_score(y_hd_train, m.predict(X_hd_train_sc)))
    hd_test_accs.append(accuracy_score(y_hd_test,  m.predict(X_hd_test_sc)))
    cv = cross_val_score(KNeighborsClassifier(n_neighbors=k),
                         X_hd_train_sc, y_hd_train, cv=5, scoring="accuracy")
    hd_cv_accs.append(cv.mean())

print(f"{'k':>5}  {'Train':>8}  {'Test':>8}  {'5-fold CV':>10}")
for k, tr, te, cv in zip(K_HD, hd_train_accs, hd_test_accs, hd_cv_accs):
    print(f"{k:>5}  {tr:>8.3f}  {te:>8.3f}  {cv:>10.3f}")

In [ ]:
# ── Step 1: compute the overfitting gap for k=1 ──────────────────────────────
# Subtract test accuracy from training accuracy for k=1
gap_k1 = None  # YOUR CODE — one line using hd_train_accs[0] and hd_test_accs[0]


# ── Step 2: identify the best k by CV accuracy ───────────────────────────────
best_k_hd = None  # YOUR CODE — use np.argmax(hd_cv_accs) to get the index, then look up the value


# ── Step 3: create a grouped bar chart comparing train, test, and CV accuracy ─
# Use seaborn and the pattern below — fill in the missing y= values

results_df = pd.DataFrame({
    "k": K_HD * 3,
    "accuracy": hd_train_accs + hd_test_accs + hd_cv_accs,
    "split": ["Train"] * len(K_HD) + ["Test"] * len(K_HD) + ["CV (5-fold)"] * len(K_HD),
})

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=results_df, x="k", y=None, hue="split", ax=ax)  # YOUR CODE — replace None with "accuracy"
ax.set_xlabel("Number of neighbors (k)")
ax.set_ylabel("Accuracy")
ax.set_title("Heart Disease: Train, Test, and CV Accuracy by k")
ax.legend(title="Split")
plt.tight_layout()
plt.show()

# ── Assertions ───────────────────────────────────────────────────────────────
assert gap_k1 is not None, \
    "Compute: gap_k1 = hd_train_accs[0] - hd_test_accs[0]"
assert gap_k1 > 0.08, \
    "k=1 should show a clear train-test gap — the model memorizes training points"
assert best_k_hd in K_HD, \
    "best_k_hd must be a value from K_HD, not an index"
assert best_k_hd != 1, \
    "k=1 overfits — CV should select a smoother model"
assert ax.patches, \
    "Complete the barplot: replace None with the column name 'accuracy'"

print(f"✓ Section C complete!")
print(f"  k=1 train-test gap : {gap_k1:.3f}")
print(f"  Best k by CV       : {best_k_hd}")

---
## Section D — Reflection

Answer each question in the string below it. There are no wrong answers. Your instructor may review these if requested.

In [ ]:
# D1 — Look at the heart disease table printed in Section C.
# At k=1, training accuracy is nearly 1.0. Why?
# What does k=1 do with each training point when predicting?

d1_answer = """
YOUR ANSWER HERE
"""

assert d1_answer.strip() != "YOUR ANSWER HERE", "Fill in your reflection for D1"
print("D1 recorded.")

In [ ]:
# D2 — In Section B and C, cross-validation picked a k that was NOT k=1,
# even though k=1 has the highest training accuracy.
# Explain in plain English why cross-validation avoids this trap.

d2_answer = """
YOUR ANSWER HERE
"""

assert d2_answer.strip() != "YOUR ANSWER HERE", "Fill in your reflection for D2"
print("D2 recorded.")

In [ ]:
# D3 — Section C used the test set to show you the final result.
# Why should the test set only be used once, at the very end?
# What goes wrong if you keep adjusting the model based on test set results?

d3_answer = """
YOUR ANSWER HERE
"""

assert d3_answer.strip() != "YOUR ANSWER HERE", "Fill in your reflection for D3"
print("D3 recorded.")